# Outlet topics and the outlet gap

Regenerates two chart-data files from the corpus CSVs in `../data/`:

* `src/data/outlet_topics.csv` — the share of each outlet's WNBA articles touching each
  topic, by year, measured two ways (full text vs. headlines only)
* `src/data/outlet_gap.json` — each player's share of each outlet's WNBA headlines,
  within the window every compared outlet covers

The patterns are the exact ones the piece's charts use.

In [ ]:
import csv, json, re
from collections import Counter, defaultdict

import pandas as pd

# the same vocabulary as the timeline: data/topic_terms.csv (see notebooks/topics.py)
from topics import patterns
PATS = patterns()
PATTERNS = {
    "polrace": re.compile(PATS["politics or race"], re.I),
    "lgbtq": re.compile(PATS["LGBTQ / sexuality"], re.I),
    "all": re.compile(PATS["politics or race or LGBTQ"], re.I),
}
GRINER = re.compile(r"griner", re.I)

csv.field_size_limit(10 ** 9)

## Topic rates by outlet and year

Two methods, never mixed on one chart: the patterns hit full articles far more often than
eight-word headlines, so `body` mode covers the full-text corpora and `headline` mode
levels the field for the slug-headline corpus (The Athletic).
Fox's own NBA coverage is the control.

In [ ]:
# output names are what the chart components key on
CORPORA_FILES = {
    "Fox News": "fox_wnba.csv",
    "Just Women's Sports": "just_womens_sports.csv",
    "Yahoo Sports (syndicated pool)": "yahoo_sports.csv",
    "ESPN": "espn.csv",
    "The Athletic": "the_athletic.csv",
    "Fox News (NBA)": "fox_nba.csv",
    "CBS Sports": "cbs_sports.csv",
    "USA Today": "usa_today.csv",
}
HEADLINE_ONLY = {"The Athletic", "CBS Sports", "USA Today"}

corpora = {}
for name, filename in CORPORA_FILES.items():
    with open(f"../data/{filename}") as fh:
        corpora[name] = [((r["date"] or "")[:10], r["headline"] or "", r["body"] or "")
                         for r in csv.DictReader(fh) if r.get("date")]
{name: len(rows) for name, rows in corpora.items()}

In [ ]:
def tally(rows, pattern, decompose=False):
    """Per year: articles, hits, and the Griner share. With decompose (the union
    topic), also the LGBTQ share — precedence griner > lgbtq > race/politics,
    so the three slices partition the hits exactly."""
    years = defaultdict(lambda: Counter())
    for date, text in rows:
        year = date[:4]
        if not year.isdigit() or year < "2019":   # the chart starts at 2019
            continue
        years[year]["n"] += 1
        if pattern.search(text):
            years[year]["hit"] += 1
            if GRINER.search(text):
                years[year]["g"] += 1
            elif decompose and PATTERNS["lgbtq"].search(text):
                years[year]["lg"] += 1
    return dict(sorted(years.items()))

records = []
for topic, pattern in PATTERNS.items():
    decompose = topic == "all"
    for method in ("body", "headline"):
        for name, rows in corpora.items():
            if method == "body" and name in HEADLINE_ONLY:
                continue
            texts = ((d, h) if method == "headline" else (d, h + " " + b)
                     for d, h, b in rows)
            for year, v in tally(texts, pattern, decompose).items():
                records.append((topic, method, name, year,
                                v["n"], v["hit"], v["g"], v["lg"]))

#### Overwrites `src/data/outlet_topics.csv`

In [ ]:
with open("../src/data/outlet_topics.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["topic", "method", "outlet", "year",
                     "articles", "hits", "griner_hits", "lgbtq_hits"])
    writer.writerows(records)
print(f"{len(records)} rows")

In [ ]:
topics = pd.read_csv("../src/data/outlet_topics.csv")
rates = topics.query("topic == 'all' and method == 'body' and articles >= 10")
rates.assign(pct=(100 * rates.hits / rates.articles).round(1)) \
     .pivot_table(index="outlet", columns="year", values="pct")

## The outlet gap

Raw counts can't compare outlets — Yahoo publishes an order of magnitude more WNBA
articles than Fox. So: each player's **share** of an outlet's WNBA headlines, Fox against
a sports-press baseline, everyone cut to the window all the compared outlets cover.
Players come from the headline network (at least 3 true Fox headlines); matching is by
surname, with guards for the surnames that belong to someone else.

In [ ]:
MIN_FOX_HEADLINES = 3
EXCLUDE = {"Layshia Clarendon"}   # in the network but not this chart
GROUPS = {
    "Yahoo + ESPN": ["Yahoo Sports", "ESPN"],
    "CBS + USA Today": ["CBS Sports", "USA Today"],
}
SPORTS_BASELINE = {o for outlets in GROUPS.values() for o in outlets}
GAP_FILES = {"Yahoo Sports": "yahoo_sports.csv", "ESPN": "espn.csv",
             "CBS Sports": "cbs_sports.csv", "USA Today": "usa_today.csv"}

NOT_HER = {
    "Sue Bird": r"larry bird|bird-magic|bird and magic",
    "A'ja Wilson": r"russell wilson|signs with wilson|wilson (?:basketball|ball|brand)",
    "Alyssa Thomas": r"lia thomas|isiah thomas",
    "Caitlin Clark": r"alysha clark",
}
NOT_HER_RX = {name: re.compile(rx, re.I) for name, rx in NOT_HER.items()}

nodes = json.load(open("../src/data/headline_network.json"))["nodes"]
players = [n["name"] for n in nodes
           if n["group"] == "players" and n["total"] >= MIN_FOX_HEADLINES
           and n["name"] not in EXCLUDE]
name_rx = {n: re.compile(r"\b" + re.escape(n.split()[-1]) + r"\b", re.I)
           for n in players}

def names_in(text):
    """Players named in this text, by surname, minus the look-alikes."""
    found = []
    for name, rx in name_rx.items():
        if not rx.search(text):
            continue
        guard = NOT_HER_RX.get(name)
        if guard and guard.search(text) and not re.search(
                re.escape(name.split()[0]), text, re.I):
            continue
        found.append(name)
    return found

players

In [ ]:
# one (date, headline) list per outlet; Fox deduplicated by headline
rows_by_outlet = defaultdict(list)
seen = set()
for row in csv.DictReader(open("../data/fox_wnba.csv")):
    headline, date = (row["headline"] or "").strip(), (row["date"] or "")[:10]
    if headline and date and headline not in seen:
        seen.add(headline)
        rows_by_outlet["Fox News"].append((date, headline))
for outlet, filename in GAP_FILES.items():
    for row in csv.DictReader(open(f"../data/{filename}")):
        headline, date = (row["headline"] or "").strip(), (row["date"] or "")[:10]
        if headline and date:
            rows_by_outlet[outlet].append((date, headline))

# the window every outlet covers: latest start, earliest end
compared = ["Fox News"] + sorted(SPORTS_BASELINE)
window = (max(min(d for d, _ in rows_by_outlet[o]) for o in compared),
          min(max(d for d, _ in rows_by_outlet[o]) for o in compared))
print(f"comparing over {window[0]} .. {window[1]}")

totals, hits = Counter(), defaultdict(Counter)
for outlet, entries in rows_by_outlet.items():
    for date, headline in entries:
        if window[0] <= date <= window[1]:
            totals[outlet] += 1
            for name in names_in(headline):
                hits[outlet][name] += 1
dict(totals)

In [ ]:
baseline_total = sum(totals[o] for o in SPORTS_BASELINE)
rows = []
for name in players:
    fox_share = hits["Fox News"][name] / totals["Fox News"]
    baseline_hits = sum(hits[o][name] for o in SPORTS_BASELINE)
    baseline_share = baseline_hits / baseline_total
    rows.append({
        "name": name,
        "fox": round(fox_share, 5),
        "foxN": hits["Fox News"][name],
        "sports": round(baseline_share, 5),
        "sportsN": baseline_hits,
        # each sports outlet on its own, then the pooled pairs
        "byOutlet": {
            **{o: {"share": round(hits[o][name] / totals[o], 5), "n": hits[o][name]}
               for o in sorted(SPORTS_BASELINE) if totals[o]},
            **{label: {"share": round(sum(hits[o][name] for o in outlets)
                                      / max(1, sum(totals[o] for o in outlets)), 5),
                       "n": sum(hits[o][name] for o in outlets)}
               for label, outlets in GROUPS.items()
               if sum(totals[o] for o in outlets)},
        },
        # how many times more (or less) of Fox's coverage she is
        "ratio": round(fox_share / baseline_share, 2) if baseline_share else None,
        "news": {},   # the CNN / MS NOW harvests live outside this repo
    })
rows.sort(key=lambda r: -r["fox"])

In [ ]:
# the same comparison over article text, where we have it
bodies = defaultdict(list)
for outlet, filename in {"Fox News": "fox_wnba.csv", **GAP_FILES}.items():
    for row in csv.DictReader(open(f"../data/{filename}")):
        text, date = (row["body"] or "").strip(), (row["date"] or "")[:10]
        if date and len(text.split()) > 80:
            bodies[outlet].append((date, text))

body_totals, body_hits = Counter(), defaultdict(Counter)
for outlet, entries in bodies.items():
    for date, text in entries:
        if window[0] <= date <= window[1]:
            body_totals[outlet] += 1
            for name in names_in(text):
                body_hits[outlet][name] += 1

for row in rows:
    name = row["name"]
    by_outlet = {}
    for label, outlets in GROUPS.items():
        denominator = sum(body_totals[o] for o in outlets)
        if denominator:
            by_outlet[label] = {
                "share": round(sum(body_hits[o][name] for o in outlets) / denominator, 5),
                "n": sum(body_hits[o][name] for o in outlets)}
    if body_totals["Fox News"] and by_outlet:
        row["body"] = {
            "fox": round(body_hits["Fox News"][name] / body_totals["Fox News"], 5),
            "foxN": body_hits["Fox News"][name],
            "byOutlet": by_outlet,
        }
dict(body_totals)

#### Overwrites `src/data/outlet_gap.json`

In [ ]:
with open("../src/data/outlet_gap.json", "w") as fh:
    fh.write(json.dumps({
        "groups": GROUPS,
        "bodyTotals": {o: n for o, n in body_totals.items()},
        "window": {"from": window[0], "to": window[1]},
        "totals": {o: totals[o] for o in totals if totals[o] >= 100},
        "baseline": sorted(SPORTS_BASELINE),
        "people": rows,
    }))

pd.DataFrame([{"player": r["name"], "fox %": round(r["fox"] * 100, 1),
               "sports %": round(r["sports"] * 100, 1), "ratio": r["ratio"]}
              for r in rows])